# 가드레일(Guardrails)

가드레일(Guardrails)은 에이전트 실행 중 주요 지점에서 콘텐츠를 검증하고 필터링하여 안전하고 규정을 준수하는 AI 애플리케이션을 구축할 수 있도록 도와줍니다. 가드레일을 통해 부적절한 입력을 차단하고, 민감한 정보를 보호하며, 출력 품질을 보장할 수 있습니다.

> 📖 **참고 문서**: 
> - [LangChain Guardrails](https://docs.langchain.com/oss/python/langchain/guardrails)
> - [Middleware Overview](https://docs.langchain.com/oss/python/langchain/middleware/overview)

## 주요 사용 사례

가드레일은 다음과 같은 상황에서 사용됩니다:

- 개인정보(PII) 유출 방지
- 프롬프트 인젝션 공격 탐지 및 차단
- 부적절하거나 유해한 콘텐츠 차단
- 비즈니스 규칙 및 규정 준수 요구사항 강제
- 출력 품질 및 정확성 검증

가드레일은 미들웨어를 사용하여 구현되며, 에이전트 시작 전, 완료 후, 또는 모델 및 도구 호출 주변에서 실행을 가로챌 수 있습니다. 이 튜토리얼에서는 다양한 가드레일 패턴과 구현 방법을 학습합니다.

![](./assets/middleware-flow.png)

## 사전 준비

가드레일 기능을 사용하기 위해 환경 변수와 LangSmith 추적을 설정합니다. 환경 변수에는 LLM 서비스 인증 정보가 포함되며, LangSmith를 통해 가드레일이 트리거된 상황을 모니터링할 수 있습니다.

아래 코드는 `.env` 파일에서 환경 변수를 로드하고, LangSmith 추적을 활성화합니다.

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging

# 환경 변수 로드
load_dotenv(override=True)

# LangSmith 추적 활성화
logging.langsmith("LangGraph-V1-Tutorial")

LangSmith 추적을 시작합니다.
[프로젝트명]
LangGraph-V1-Tutorial


## 가드레일의 두 가지 접근 방식

가드레일은 구현 방식에 따라 크게 두 가지로 나눌 수 있습니다. 각 접근 방식은 장단점이 있으며, 실제 애플리케이션에서는 두 가지를 조합하여 사용하는 것이 일반적입니다.

### 1. 결정론적 가드레일 (Deterministic)

정규 표현식, 키워드 매칭, 명시적 검사 등 규칙 기반 로직을 사용합니다. 빠르고 예측 가능하며 비용 효율적이지만, 미묘한 위반 사항을 놓칠 수 있습니다. 금지어 필터링, PII 패턴 탐지 등에 적합합니다.

### 2. 모델 기반 가드레일 (Model-based)

LLM 또는 분류기를 사용하여 의미론적 이해로 콘텐츠를 평가합니다. 규칙이 놓치는 미묘한 문제를 포착할 수 있지만, 느리고 비용이 더 많이 듭니다. 감정 분석, 안전성 평가, 품질 검증 등에 적합합니다.

## 내장 가드레일

LangChain은 자주 사용되는 가드레일 패턴을 내장 미들웨어로 제공합니다. 별도의 구현 없이 바로 사용할 수 있어 빠르게 보안 기능을 추가할 수 있습니다.

> 📖 **참고 문서**: [Built-in Middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in)

### PII 탐지

개인 식별 정보(PII)를 대화에서 탐지하고 처리하기 위한 `PIIMiddleware`를 제공합니다. 이메일, 신용카드 번호, 전화번호 등 다양한 PII 유형을 지원하며, 다음과 같은 처리 전략을 제공합니다:

| 전략 | 설명 | 예시 |
|------|------|------|
| `redact` | `[REDACTED_TYPE]`으로 대체하여 완전히 제거 | `john@example.com` → `[REDACTED_EMAIL]` |
| `mask` | 부분적으로 가림 | `4532-1234-5678-9010` → `****-9010` |
| `hash` | 결정론적 해시로 대체하여 추적 가능성 유지 | `192.168.1.1` → `a1b2c3d4` |
| `block` | 탐지 시 예외를 발생시켜 처리 중단 | 처리 중단 |

아래 코드는 사용자 입력에서 이메일을 제거하고 신용카드를 마스킹하는 에이전트를 생성합니다.

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_teddynote.messages import invoke_graph


@tool
def customer_service_tool(query: str) -> str:
    """Handle customer service queries."""
    # 고객 서비스 쿼리 처리
    return f"Processing customer query: {query}"


# OpenAI 키 사용 시 gpt-4o-mini 등으로 변경하세요
model = init_chat_model("claude-sonnet-4-5")

# PII 보호가 적용된 에이전트 생성
agent = create_agent(
    model=model,
    tools=[customer_service_tool],
    middleware=[
        # 사용자 입력에서 이메일 수정 (완전히 제거)
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # 사용자 입력에서 신용카드 마스킹 (부분 가림)
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        ],
)

# PII가 포함된 메시지 테스트 - 이메일은 [REDACTED_EMAIL], 신용카드는 ****-6467 형태로 변환됨
invoke_graph(
    agent,
    {"messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and credit card is 4539-1488-0343-6467."
    }]
})


🔄 Node: PIIMiddleware[email].before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Human Message =================================

My email is [REDACTED_EMAIL] and credit card is 4539-1488-0343-6467.

🔄 Node: PIIMiddleware[credit_card].before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Human Message =================================

My email is [REDACTED_EMAIL] and credit card is ****-****-****-6467.

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

I appreciate you sharing that information, but I want to let you know that I don't need your email or credit card details for our conversation. 

If you have a customer service question or issue you'd like help with, please feel free to describe it and I'll be happy to assist you. You can keep your personal and financial information private unless 

### 커스텀 PII 탐지기

내장 PII 타입 외에도 정규 표현식을 사용하여 커스텀 PII 패턴을 정의할 수 있습니다. API 키, 특정 형식의 ID, 지역별 전화번호 패턴 등 비즈니스 요구사항에 맞는 패턴을 추가할 수 있습니다.

아래 코드는 API 키 패턴과 한국 전화번호 패턴을 탐지하는 커스텀 PII 미들웨어를 생성합니다.

In [8]:
from langchain.agents.middleware import PIIMiddleware

# 커스텀 PII 탐지기가 적용된 에이전트 생성
agent = create_agent(
    model=model,
    tools=[customer_service_tool],
    middleware=[
        # API 키 탐지 - 발견 시 에러 발생 (처리 중단)
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",  # 커스텀 정규 표현식
            strategy="block",
            apply_to_input=True,
        ),
        # 한국 전화번호 패턴 탐지 (010-XXXX-XXXX 형식)
        PIIMiddleware(
            "phone_number",
            detector=r"010-\d{4}-\d{4}",
            strategy="redact",
            apply_to_input=True,
        ),
    ],
)

# 전화번호가 포함된 메시지 테스트 - 전화번호는 [REDACTED_PHONE_NUMBER]로 변환됨
invoke_graph(
    agent,
    {"messages": [{
        "role": "user",
        "content": "My phone is 010-1234-5678"
    }]
})


🔄 Node: PIIMiddleware[api_key].before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: PIIMiddleware[phone_number].before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Human Message =================================

My phone is [REDACTED_PHONE_NUMBER]

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

I see you've shared a phone number, but I'm not sure what you'd like me to help you with. Could you please clarify:

- Are you trying to update your phone number for an account?
- Do you need to verify your phone number?
- Are you reporting an issue related to your phone service?
- Is there something else you need assistance with?

Please let me know what you're looking to accomplish, and I'll be happy to help!

🔄 Node: PIIMiddleware[phone_number].after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: PIIMi

In [9]:
# API 키가 포함된 메시지 테스트 - 처리 중단 및 에러 발생
try:
    result = agent.invoke({
        "messages": [{"role": "user", "content": "My API key is sk-1234567890abcdef1234567890abcdef"}]
    })
except Exception as e:
    print(f"Error: {e}")

Error: Detected 1 instance(s) of api_key in text content


### 내장 PII 타입

LangChain은 자주 사용되는 PII 패턴을 내장 타입으로 제공합니다. 이러한 타입은 검증된 패턴을 사용하므로 신뢰성이 높습니다:

| 타입 | 설명 | 예시 패턴 |
|------|------|----------|
| `email` | 이메일 주소 | `user@domain.com` |
| `credit_card` | 신용카드 번호 (Luhn 알고리즘 검증 포함) | `4532-1234-5678-9010` |
| `ip` | IP 주소 (IPv4, IPv6) | `192.168.1.1`, `::1` |
| `mac_address` | MAC 주소 | `00:1A:2B:3C:4D:5E` |
| `url` | URL | `https://example.com` |

아래 코드는 여러 내장 PII 타입을 동시에 적용하는 에이전트를 생성합니다.

In [10]:
# 여러 PII 타입 동시 적용
agent = create_agent(
    model=model,
    tools=[customer_service_tool],
    middleware=[
        # 이메일 주소 수정
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        # IP 주소 해시 처리 (추적 가능성 유지)
        PIIMiddleware("ip", strategy="hash", apply_to_input=True),
        # URL 수정
        PIIMiddleware("url", strategy="redact", apply_to_input=True),
    ],
)

# 여러 PII 유형이 포함된 메시지 테스트 - 각 PII 유형에 맞는 전략이 적용됨
invoke_graph(
    agent,
    {"messages": [{
        "role": "user",
        "content": "Contact me at user@test.com from IP 192.168.1.1 or visit https://example.com"
    }]
})



🔄 Node: PIIMiddleware[email].before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Human Message =================================

Contact me at [REDACTED_EMAIL] from IP 192.168.1.1 or visit https://example.com

🔄 Node: PIIMiddleware[ip].before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Human Message =================================

Contact me at [REDACTED_EMAIL] from IP <ip_hash:c5eb5a4c> or visit https://example.com

🔄 Node: PIIMiddleware[url].before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Human Message =================================

Contact me at [REDACTED_EMAIL] from IP <ip_hash:c5eb5a4c> or visit [REDACTED_URL]

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

I appreciate you reaching out, but I need to let you know that I cannot 

## Human-in-the-Loop

민감한 작업을 실행하기 전에 사람의 승인을 요구하는 것은 가장 효과적인 가드레일 중 하나입니다. `HumanInTheLoopMiddleware`를 사용하면 특정 도구 호출 전에 실행을 일시 중지하고 사람의 결정을 기다릴 수 있습니다. 이 기능에 대한 자세한 내용은 Human-in-the-Loop 튜토리얼을 참조하세요.

**Human-in-the-Loop이 유용한 경우:**
- 금융 거래 및 송금
- 프로덕션 데이터 삭제 또는 수정
- 외부 당사자에게 통신 전송
- 비즈니스에 중요한 영향을 미치는 모든 작업

아래 코드는 이메일 전송과 데이터베이스 삭제 작업에 대해 사람의 승인을 요구하는 에이전트를 생성합니다.

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.tools import tool


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    # 정보 검색 (안전한 작업)
    return f"Search results for: {query}"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Send an email. This is a sensitive operation."""
    # 이메일 전송 (민감한 작업 - 승인 필요)
    return f"Email sent to {recipient}"


@tool
def delete_database_tool(database_name: str) -> str:
    """Delete a database. This is a critical operation."""
    # 데이터베이스 삭제 (위험한 작업 - 승인 필요)
    return f"Database {database_name} deleted"


# Human-in-the-Loop 미들웨어가 적용된 에이전트 생성
agent = create_agent(
    model=model,
    tools=[search_tool, send_email_tool, delete_database_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # 민감한 작업에 대해 승인 필요
                "send_email_tool": True,
                "delete_database_tool": True,
                # 안전한 작업은 자동 승인
                "search_tool": False,
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # 상태 지속성 필요 (재개를 위해)
)

# thread_id 필요 (대화 상태 추적용)
config = {"configurable": {"thread_id": "some_id"}}

# 이메일 전송 전 일시 중지됨
invoke_graph(
    agent,
    {"messages": [{
        "role": "user", 
        "content": "Send an email to team@company.com with subject 'Weekly Update' and body 'Please review the latest changes."
        }]
        },
    config=config
)

print("에이전트가 승인을 위해 일시 중지되었습니다. 승인 중...")

# 승인 후 재개 - Command를 사용하여 승인 결정 전달
invoke_graph(
    agent,
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config  # 동일한 thread_id로 재개
)


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

[{'id': 'toolu_01NkvvM4GQ1zn9ERh86UNjQK', 'input': {'recipient': 'team@company.com', 'subject': 'Weekly Update', 'body': 'Please review the latest changes.'}, 'name': 'send_email_tool', 'type': 'tool_use'}]
Tool Calls:
  send_email_tool (toolu_01NkvvM4GQ1zn9ERh86UNjQK)
 Call ID: toolu_01NkvvM4GQ1zn9ERh86UNjQK
  Args:
    recipient: team@company.com
    subject: Weekly Update
    body: Please review the latest changes.

🔄 Node: __interrupt__ 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
Interrupt(value={'action_requests': [{'name': 'send_email_tool', 'args': {'recipient': 'team@company.com', 'subject': 'Weekly Update', 'body': 'Please review the latest changes.'}, 'description': "Tool execution requires approval\n\nTool: send_email_tool\nArgs: {'recipient': 'team@company.com', 'subject': 'Weekly Update', 'body': 'Please review the l

## 커스텀 가드레일

내장 가드레일로 충분하지 않은 경우 에이전트 실행 전후에 실행되는 커스텀 미들웨어를 생성할 수 있습니다. `@before_agent`, `@after_agent` 데코레이터 또는 `AgentMiddleware` 클래스를 사용하여 비즈니스 요구사항에 맞는 가드레일을 구현합니다.

> 📖 **참고 문서**: [Custom Middleware](https://docs.langchain.com/oss/python/langchain/middleware/custom)

### 두 가지 구현 방식

| 방식 | 사용 시기 | 장점 |
|------|----------|------|
| 데코레이터 (`@before_agent`, `@after_agent`) | 단순한 검증 로직 | 간결한 코드 |
| 클래스 (`AgentMiddleware` 상속) | 설정 가능한 매개변수, 상태 유지 | 재사용성, 여러 훅 조합 |

### Before Agent 가드레일

`@before_agent` 데코레이터를 사용하면 각 호출 시작 시 요청을 검증할 수 있습니다. 인증 확인, 속도 제한, 금지어 필터링 등 처리가 시작되기 전에 부적절한 요청을 차단하는 데 유용합니다. `can_jump_to=["end"]` 옵션을 사용하면 가드레일이 트리거될 때 에이전트 실행을 즉시 종료할 수 있습니다.

아래 코드는 금지된 키워드가 포함된 요청을 처리 전에 차단하는 결정론적 가드레일을 구현합니다.

In [18]:
from typing import Any
from langchain.agents.middleware import before_agent, AgentState


@before_agent(can_jump_to=["end"])
def content_filter(state: AgentState, runtime) -> dict[str, Any] | None:
    """결정론적 가드레일: 금지된 키워드가 포함된 요청 차단
    
    에이전트 실행 전에 호출되어 부적절한 콘텐츠를 사전에 필터링합니다.
    """
    # 금지된 키워드 목록
    banned_keywords = ["hack", "exploit", "malware", "해킹", "악성코드"]
    
    # 첫 번째 사용자 메시지 가져오기
    if not state["messages"]:
        return None

    first_message = state["messages"][0]
    # human 타입 메시지만 검사
    if first_message.type != "human":
        return None

    # 소문자로 변환하여 대소문자 구분 없이 검사
    content = first_message.content.lower()

    # 금지된 키워드 확인
    for keyword in banned_keywords:
        if keyword in content:
            # 처리 전에 실행 차단 - jump_to="end"로 에이전트 종료
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "부적절한 콘텐츠가 포함된 요청은 처리할 수 없습니다. 요청을 다시 작성해주세요."
                }],
                "jump_to": "end"
            }

    # None 반환 시 정상 진행
    return None


# 커스텀 가드레일이 적용된 에이전트 생성
agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[content_filter],
)

# 이 요청은 "hack" 키워드로 인해 처리 전에 차단됨
invoke_graph(
    agent,
    {"messages": [{
        "role": "user",
        "content": "How do I hack into a database?"
    }]
})



🔄 Node: content_filter.before_agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
{'role': 'assistant', 'content': '부적절한 콘텐츠가 포함된 요청은 처리할 수 없습니다. 요청을 다시 작성해주세요.'}
jump_to:
end


### 클래스 기반 Before Agent 가드레일

더 복잡한 로직이 필요하거나 설정 가능한 매개변수가 있는 경우 `AgentMiddleware` 클래스를 상속받아 구현할 수 있습니다. 클래스 기반 접근 방식은 상태를 유지하거나 여러 후크를 함께 사용해야 할 때 유용합니다.

아래 코드는 금지어 목록을 매개변수로 받는 재사용 가능한 `ContentFilterMiddleware` 클래스를 구현합니다.

In [ ]:
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from typing import Any


class ContentFilterMiddleware(AgentMiddleware):
    """결정론적 가드레일: 금지된 키워드가 포함된 요청 차단
    
    클래스 기반 구현으로 설정 가능한 매개변수와 상태 유지가 가능합니다.
    여러 후크를 하나의 클래스에서 관리할 수 있어 복잡한 로직에 적합합니다.
    """

    def __init__(self, banned_keywords: list[str]):
        """
        Args:
            banned_keywords: 차단할 키워드 목록
        """
        super().__init__()
        # 대소문자 구분 없이 검사하기 위해 소문자로 저장
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    # hook_config 데코레이터로 후크 설정 - can_jump_to: 훅이 점프할 수 있는 대상 목록
    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime) -> dict[str, Any] | None:
        """에이전트 실행 전에 호출되는 훅"""
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        # human 타입 메시지만 검사
        if first_message.type != "human":
            return None

        # 소문자로 변환하여 대소문자 구분 없이 검사
        content = first_message.content.lower()

        # 금지된 키워드가 포함되어 있으면 차단
        for keyword in self.banned_keywords:
            if keyword in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": "부적절한 콘텐츠가 포함된 요청은 처리할 수 없습니다."
                    }],
                    "jump_to": "end"
                }

        return None


# 클래스 기반 가드레일이 적용된 에이전트 생성
agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        # 금지어 목록을 매개변수로 전달
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "해킹"]
        ),
    ],
)

# 한국어 "해킹" 키워드로 인해 차단됨
invoke_graph(
    agent,
    {"messages": [{
        "role": "user",
        "content": "데이터베이스를 해킹하는 방법은?"
    }]
})


🔄 Node: ContentFilterMiddleware.before_agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
{'role': 'assistant', 'content': '부적절한 콘텐츠가 포함된 요청은 처리할 수 없습니다.'}
jump_to:
end


### After Agent 가드레일

`@after_agent` 데코레이터를 사용하면 사용자에게 반환하기 전에 최종 출력을 검증할 수 있습니다. 모델 기반 안전성 검사, 품질 검증, 규정 준수 스캔 등 전체 에이전트 응답을 평가하는 데 유용합니다. 입력 검증보다 비용이 높을 수 있지만, 더 정교한 검사가 가능합니다.

아래 코드는 LLM을 사용하여 에이전트 응답의 안전성을 평가하는 모델 기반 가드레일을 구현합니다.

In [23]:
from langchain.agents.middleware import after_agent, AgentState
from langchain_core.messages import AIMessage
from langchain.chat_models import init_chat_model
from typing import Any

# 안전성 평가를 위한 별도 모델 (비용 효율적인 모델 사용)
# OpenAI 키 사용 시 gpt-4o-mini 등으로 변경하세요
safety_model = init_chat_model("claude-sonnet-4-5")


@after_agent(can_jump_to=["end"])
def safety_guardrail(state: AgentState, runtime) -> dict[str, Any] | None:
    """모델 기반 가드레일: LLM을 사용하여 응답 안전성 평가
    
    에이전트 실행 완료 후 호출되어 최종 응답을 검증합니다.
    입력 검증보다 비용이 높지만, 더 정교한 검사가 가능합니다.
    """
    # 메시지가 없으면 건너뜀
    if not state["messages"]:
        return None

    # 최종 AI 응답 가져오기
    last_message = state["messages"][-1]
    # AI 메시지가 아니면 건너뜀
    if not isinstance(last_message, AIMessage):
        return None

    # 모델을 사용하여 안전성 평가
    safety_prompt = f"""이 응답이 안전하고 적절한지 평가하세요.
    'SAFE' 또는 'UNSAFE'로만 응답하세요.

    응답: {last_message.content}"""

    # 안전성 모델로 평가 실행
    result = safety_model.invoke([{"role": "user", "content": safety_prompt}])

    # UNSAFE 판정 시 응답 차단
    if "UNSAFE" in result.content:
        return {
            "messages": [{
                "role": "assistant",
                "content": "해당 응답을 제공할 수 없습니다. 요청을 다시 작성해주세요."
            }],
            "jump_to": "end"
        }

    # SAFE 판정 시 정상 진행
    return {
        "messages": [{
            "role": "assistant",
            "content": f"[Safety Check] 평가 결과: {result.content}"
        }]
    }


# 안전 가드레일이 적용된 에이전트 생성
agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[safety_guardrail],
)

# 안전한 요청 테스트
invoke_graph(
    agent,
    {"messages": [{
        "role": "user",
        "content": "Tell me about Python programming"
    }]
})


🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

[{'id': 'toolu_01R78Lko133Ype2nvfBG4S72', 'input': {'query': 'Python programming language'}, 'name': 'search_tool', 'type': 'tool_use'}]
Tool Calls:
  search_tool (toolu_01R78Lko133Ype2nvfBG4S72)
 Call ID: toolu_01R78Lko133Ype2nvfBG4S72
  Args:
    query: Python programming language

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: search_tool

Search results for: Python programming language

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

Based on the search and general knowledge, here's an overview of Python programming:

## Python Programming Language

**What is Python?**
Python is a high-level, interpreted programming language creat

## 여러 가드레일 결합

미들웨어 배열에 여러 가드레일을 추가하여 계층화된 보호(Defense in Depth)를 구축할 수 있습니다. 가드레일은 배열 순서대로 실행되며, 빠른 결정론적 검사를 먼저, 느린 모델 기반 검사를 나중에 배치하면 불필요한 비용을 줄일 수 있습니다.

### 권장 가드레일 순서

```
1. 속도 제한 (Rate Limiting) - 즉시 차단
2. 결정론적 입력 필터 (키워드, 정규표현식) - 밀리초 단위
3. PII 보호 (입력) - 밀리초 단위
4. Human-in-the-Loop - 민감한 작업 승인
5. PII 보호 (출력) - 밀리초 단위
6. 모델 기반 검증 - 초 단위 (비용 발생)
```

아래 코드는 입력 필터, PII 보호, Human-in-the-Loop, 안전성 검사를 계층화하여 적용하는 에이전트를 생성합니다.

In [25]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# 다층 보호(Defense in Depth)가 적용된 에이전트 생성
agent = create_agent(
    model=model,
    tools=[search_tool, send_email_tool],
    middleware=[
        # 계층 1: 결정론적 입력 필터 (before agent)
        # 빠른 규칙 기반 검사로 명백한 위반을 조기에 차단
        ContentFilterMiddleware(banned_keywords=["hack", "exploit"]),

        # 계층 2: PII 보호 (before and after model)
        # 입력과 출력 모두에서 이메일 주소 수정
        PIIMiddleware("email", strategy="redact", apply_to_input=True, apply_to_output=True),

        # 계층 3: 민감한 도구에 대한 사람의 승인
        # 이메일 전송 전 반드시 승인 필요
        HumanInTheLoopMiddleware(interrupt_on={"send_email_tool": True}),

        # 계층 4: 모델 기반 안전성 검사 (after agent)
        # 최종 응답을 LLM으로 검증
        safety_guardrail,
    ],
    checkpointer=InMemorySaver(),  # Human-in-the-Loop을 위해 필요
)

# 대화 상태 추적을 위한 config
config = {"configurable": {"thread_id": "secure_thread"}}

# 모든 계층의 가드레일을 통과해야 함
invoke_graph(
    agent,
    {"messages": [{
        "role": "user",
        "content": "Send an email to team@example.com"}
    ]},
    config=config
)


🔄 Node: ContentFilterMiddleware.before_agent 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: PIIMiddleware[email].before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Human Message =================================

Send an email to [REDACTED_EMAIL]

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

I'd be happy to help you send an email to that address. However, I need a few more details to complete the email:

1. **Subject**: What should the subject line be?
2. **Body**: What would you like to say in the email?

Please provide these details and I'll send the email for you.

🔄 Node: HumanInTheLoopMiddleware.after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: PIIMiddleware[email].after_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: safety_guardrail.after_agent 🔄
- - - - - - - - - - - - 

## 종합 예제: 보안 고객 서비스 에이전트

이 섹션에서는 지금까지 학습한 가드레일 기술을 결합하여 실용적인 고객 서비스 에이전트를 구현합니다. 속도 제한, 콘텐츠 필터링, PII 보호, 출력 품질 검증을 모두 적용하여 안전하고 신뢰할 수 있는 에이전트를 구축합니다.

아래 코드는 고객 조회, 환불 처리, 알림 전송 기능을 가진 보안 고객 서비스 에이전트를 구현합니다.

In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, AgentMiddleware, after_agent, hook_config
from langchain.tools import tool
from langchain_core.messages import AIMessage
from typing import Any


@tool
def lookup_customer(customer_id: str) -> str:
    """Look up customer information."""
    # 고객 정보 조회
    return f"Customer {customer_id}: John Doe, Premium member"


@tool
def process_refund(order_id: str, amount: float) -> str:
    """Process a refund for an order."""
    # 환불 처리
    return f"Refund of ${amount} processed for order {order_id}"


@tool
def send_notification(customer_id: str, message: str) -> str:
    """Send notification to customer."""
    # 고객에게 알림 전송
    return f"Notification sent to customer {customer_id}"


# 속도 제한 가드레일 (Rate Limiting)
class RateLimitMiddleware(AgentMiddleware):
    """요청 속도 제한 가드레일
    
    짧은 시간 내 너무 많은 요청을 차단하여 시스템을 보호합니다.
    실제 프로덕션에서는 Redis 등을 사용한 분산 카운터를 사용해야 합니다.
    """
    
    def __init__(self, max_requests: int = 10):
        """
        Args:
            max_requests: 허용되는 최대 요청 수
        """
        super().__init__()
        self.request_count = 0
        self.max_requests = max_requests

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state, runtime) -> dict[str, Any] | None:
        """에이전트 실행 전 요청 수 확인"""
        self.request_count += 1
        # 요청 한도 초과 시 차단
        if self.request_count > self.max_requests:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "요청 한도를 초과했습니다. 잠시 후 다시 시도해주세요."
                }],
                "jump_to": "end"
            }
        return None


# 출력 품질 검증 가드레일
@after_agent
def validate_output(state, runtime) -> dict[str, Any] | None:
    """응답이 충분히 도움이 되는지 확인
    
    응답 품질을 검증하고 필요시 경고를 출력합니다.
    실제 프로덕션에서는 더 정교한 품질 검사를 구현할 수 있습니다.
    """
    if not state["messages"]:
        return None
    
    last_message = state["messages"][-1]
    # AI 메시지인 경우에만 검사
    if isinstance(last_message, AIMessage):
        # 응답이 너무 짧으면 경고 (차단하지는 않음)
        if len(last_message.content) < 20:
            print("Warning: Response may be too brief")
    
    # None 반환 시 정상 진행 (차단하지 않음)
    return None


# 보안 고객 서비스 에이전트 생성
secure_agent = create_agent(
    model=model,
    tools=[lookup_customer, process_refund, send_notification],
    middleware=[
        # 1. 입력 검증 단계 (빠른 검사)
        RateLimitMiddleware(max_requests=100),  # 속도 제한
        ContentFilterMiddleware(banned_keywords=["hack", "fraud"]),  # 금지어 필터
        
        # 2. PII 보호 단계
        PIIMiddleware("email", strategy="redact", apply_to_input=True, apply_to_output=True),  # 입력 및 출력 이메일 수정
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),  # 신용카드 마스킹
        
        # 3. 출력 검증 단계
        validate_output,  # 품질 검증
    ],
)

# === 테스트 1: 정상 요청 ===
print("=== Test 1: Normal query ===")
result = secure_agent.invoke({
    "messages": [{"role": "user", "content": "Look up customer CUST123"}]
})
print(result["messages"][-1].content)

# === 테스트 2: PII가 포함된 요청 ===
print("\n=== Test 2: Query with PII ===")
result = secure_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Customer email is john@example.com and card is 4532-1234-5678-9010"
    }]
})
print(result["messages"][-1].content)

# === 테스트 3: 차단되는 콘텐츠 ===
print("\n=== Test 3: Blocked content ===")
result = secure_agent.invoke({
    "messages": [{"role": "user", "content": "How to hack the system?"}]
})
print(result["messages"][-1].content)

=== Test 1: Normal query ===
I've found the customer information:

**Customer ID:** CUST123  
**Name:** John Doe  
**Status:** Premium member

Is there anything else you'd like me to help you with regarding this customer?

=== Test 2: Query with PII ===
I appreciate you providing that information, but I want to let you know that I don't have any tools available that require email addresses or credit card numbers. 

Could you please let me know what you'd like help with today? For example:
- Looking up customer information (I would need a customer ID)
- Processing a refund (I would need an order ID and refund amount)
- Sending a notification to a customer (I would need a customer ID and the message to send)

Please share what you need assistance with and provide the relevant information like customer ID or order ID.

=== Test 3: Blocked content ===
부적절한 콘텐츠가 포함된 요청은 처리할 수 없습니다.


## 가드레일 설계 모범 사례

효과적인 가드레일 시스템을 구축하기 위한 핵심 원칙을 정리합니다. 이러한 원칙을 따르면 안전하면서도 사용자 경험을 해치지 않는 가드레일을 구현할 수 있습니다.

### 1. 계층화된 방어 (Defense in Depth)

여러 가드레일을 결합하여 다층 보호를 구현합니다. 한 계층이 실패해도 다른 계층이 보호할 수 있도록 설계합니다. 빠른 규칙 기반 필터를 먼저 배치하고, 느리지만 정확한 모델 기반 검증을 나중에 배치합니다.

| 순서 | 계층 | 속도 / 비용 | 역할 |
|:----:|------|------------|------|
| 1 | **결정론적 필터** | 빠름 / 저비용 | 키워드 매칭, 정규표현식으로 명백한 위반을 조기 차단 |
| 2 | **PII 보호** | 빠름 / 저비용 | 이메일, 신용카드, 전화번호 등 개인정보 탐지 및 처리 |
| 3 | **모델 기반 검증** | 느림 / 고비용 | LLM을 사용한 의미론적 안전성 평가 |

아래 코드는 다층 보호를 적용하는 모범 사례 예제입니다.

In [28]:
# 좋은 예: 다층 보호 (Defense in Depth)
agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        # 입력 단계: 빠른 규칙 기반 필터 (밀리초 단위)
        # 명백한 위반을 조기에 차단하여 불필요한 비용 방지
        ContentFilterMiddleware(banned_keywords=["hack"]),
        PIIMiddleware("email", strategy="redact"),
        
        # 출력 단계: 느리지만 정확한 모델 기반 검증 (초 단위)
        # 입력 필터를 통과한 요청만 검증하므로 비용 절감
        safety_guardrail,
    ],
)

### 2. 성능 고려사항

가드레일 순서는 성능에 큰 영향을 미칩니다. 빠른 검사(정규 표현식, 키워드 매칭)를 먼저 실행하여 명백한 위반을 조기에 차단하면 느린 검사(LLM 호출)를 실행할 필요가 없어 비용을 절감할 수 있습니다.

아래 코드는 올바른 가드레일 순서(빠른 검사 우선)와 잘못된 순서(느린 검사 우선)를 비교합니다.

In [29]:
# 좋은 예: 빠른 검사가 먼저 (성능 최적화)
agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        # 1순위: 빠름 (정규표현식, 키워드 매칭)
        # 명백한 위반을 조기에 차단
        ContentFilterMiddleware(banned_keywords=["hack"]),
        # 2순위: 느림 (LLM 호출 필요)
        # 입력 필터를 통과한 요청만 검증
        safety_guardrail,
    ],
)

# 나쁜 예: 느린 검사가 먼저 (비용 낭비)
# 모든 요청에 대해 LLM 호출이 발생하여 비용 증가
# agent = create_agent(
#     model=model,
#     tools=[search_tool],
#     middleware=[
#         safety_guardrail,  # 느림 - 모든 요청에 대해 실행됨
#         ContentFilterMiddleware(banned_keywords=["hack"]),  # 빠름 - 이미 비용 발생 후 실행
#     ],
# )

### 3. 명확한 에러 메시지

사용자에게 요청이 차단된 이유를 명확하게 알려주세요. 모호한 에러 메시지는 사용자 경험을 저해하고, 같은 문제가 반복될 수 있습니다. 가능하면 대안이나 다음 단계도 안내합니다.

아래 코드는 구체적인 피드백을 제공하는 가드레일 예제입니다.

In [30]:
from langchain.agents.middleware import before_agent, AgentState
from typing import Any


@before_agent(can_jump_to=["end"])
def clear_error_messages(state: AgentState, runtime) -> dict[str, Any] | None:
    """명확한 에러 메시지를 제공하는 가드레일
    
    모호한 에러 메시지 대신 구체적인 이유와 대안을 제공하여
    사용자 경험을 개선합니다.
    """
    if not state["messages"]:
        return None

    first_message = state["messages"][0]
    # human 타입 메시지만 검사
    if first_message.type != "human":
        return None

    # 소문자로 변환하여 검사
    content = first_message.content.lower()

    # 비밀번호 관련 질문에 대한 구체적인 피드백
    if "password" in content or "비밀번호" in content:
        return {
            "messages": [{
                "role": "assistant",
                "content": "보안상의 이유로 비밀번호 관련 질문은 처리할 수 없습니다. "
                           "계정 설정에서 비밀번호를 재설정해주세요."
            }],
            "jump_to": "end"
        }

    # 정상 진행
    return None

### 4. 로깅 및 모니터링

가드레일이 트리거될 때 로그를 남겨 보안 위협을 추적하고 분석하세요. 로그에는 타임스탬프, 사용자 정보, 차단 사유, 원본 콘텐츠(민감 정보 제외) 등을 포함합니다. 이 데이터를 기반으로 가드레일 정책을 지속적으로 개선할 수 있습니다.

아래 코드는 보안 이벤트를 로깅하는 가드레일 예제입니다.

In [31]:
import logging
from langchain.agents.middleware import before_agent, AgentState
from typing import Any

# 로거 설정
logger = logging.getLogger(__name__)


@before_agent(can_jump_to=["end"])
def monitored_guardrail(state: AgentState, runtime) -> dict[str, Any] | None:
    """로깅이 포함된 가드레일
    
    보안 이벤트를 로깅하여 위협을 추적하고 분석할 수 있도록 합니다.
    로그 데이터를 기반으로 가드레일 정책을 지속적으로 개선할 수 있습니다.
    """
    if not state["messages"]:
        return None

    first_message = state["messages"][0]
    # human 타입 메시지만 검사
    if first_message.type != "human":
        return None

    # 소문자로 변환하여 검사
    content = first_message.content.lower()

    # 보안 위반 탐지 시 로깅 후 차단
    if "hack" in content:
        # 보안 이벤트 로깅 (민감 정보는 처음 50자만)
        # 실제 프로덕션에서는 타임스탬프, 사용자 ID 등 추가 정보 포함
        logger.warning(f"Security violation detected: {content[:50]}...")
        
        return {
            "messages": [{
                "role": "assistant",
                "content": "요청이 보안 정책에 위배됩니다."
            }],
            "jump_to": "end"
        }

    # 정상 진행
    return None

## 정리

이 튜토리얼에서는 LangChain의 가드레일 시스템을 사용하여 안전하고 규정을 준수하는 AI 애플리케이션을 구축하는 방법을 학습했습니다.

### 핵심 개념 요약

| 개념 | 설명 |
|------|------|
| **가드레일** | 에이전트 실행 중 콘텐츠를 검증하고 필터링하는 보안 메커니즘 |
| **결정론적 가드레일** | 정규표현식, 키워드 매칭 등 규칙 기반 (빠름, 저비용) |
| **모델 기반 가드레일** | LLM을 사용한 의미론적 검증 (느림, 고비용, 정교함) |
| **PIIMiddleware** | 개인정보(이메일, 신용카드 등) 탐지 및 보호 |
| **HumanInTheLoopMiddleware** | 민감한 작업 전 사람의 승인 요구 |
| **@before_agent** | 에이전트 실행 전 입력 검증 |
| **@after_agent** | 에이전트 실행 후 출력 검증 |
| **AgentMiddleware** | 클래스 기반 커스텀 미들웨어 구현 |

### 모범 사례 체크리스트

- [ ] 계층화된 방어(Defense in Depth) 적용
- [ ] 빠른 검사를 먼저, 느린 검사를 나중에 배치
- [ ] 명확한 에러 메시지로 사용자 경험 개선
- [ ] 보안 이벤트 로깅 및 모니터링 구현
- [ ] 민감한 작업에 Human-in-the-Loop 적용
- [ ] 입력과 출력 모두에 PII 보호 적용

### 추가 학습 자료

가드레일과 미들웨어에 대해 더 자세히 알아보려면 아래 공식 문서를 참고하세요:

> 📖 **참고 문서**:
> - [LangChain Guardrails](https://docs.langchain.com/oss/python/langchain/guardrails)
> - [Middleware Overview](https://docs.langchain.com/oss/python/langchain/middleware/overview)
> - [Built-in Middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in)
> - [Custom Middleware](https://docs.langchain.com/oss/python/langchain/middleware/custom)